# Phase 12 - Experten statt Einheiten

**Braucht eine A100**, ~20 min.

Die Eichung hat die Eichmarke bestaetigt und das Instrument verworfen:

| | k/n | Rate |
|---|---|---|
| JP `each service's Japanese name` | 47/48 | 97.9 % japanisch |
| NEU `each service's name` | 0/48 | 0.0 % |
| 512 bestgewaehlte Einheiten aus | 48/48 | 100.0 %, p=1.0000 |
| 512 zufaellige Einheiten aus | 48/48 | 100.0 %, p=1.0000 |
| Leiter bis 1024 zufaellige | | alles still, Antworten fluessig |

An der gemessenen Stelle waren **985 der 4096 dort aktiven Einheiten genullt** - ein
Viertel der aktiven Zwischenschicht - und das Verhalten ruehrte sich nicht. Die Auswahl
war dabei *nicht* wertlos: ihre Einheiten bewegten die Logits rund dreifach staerker als
zufaellige gleicher Zahl (3.66 gegen 1.27). Sie hatte Signal, nur keine Verhaltensfolge.

## Der Denkfehler, den diese Zelle behebt

```
aktive Paare: JP 320 | NEU 320 | in BEIDEN 278
```

**42 (Schicht, Experte)-Paare feuern im JP-Arm und im NEU-Arm nicht.** Der Unterschied
zwischen "antworte japanisch" und "antworte englisch" liegt zu einem guten Teil darin,
*welche Experten ueberhaupt laufen*. Der Eingriff hat daran nichts geaendert - im
Gegenteil: die Beschraenkung auf gemeinsame Paare, der richtige Fix gegen die
strukturellen Nullen, hat die 42 unterscheidenden Experten **systematisch
ausgeschlossen**. Gesucht wurde in der Schnittmenge, waehrend die Information in der
Differenz liegt.

## Ein anderes Instrument, nicht eine andere Auswahl

Statt Gewichtszeilen zu nullen wird der **Router-Anteil ganzer Experten** auf null
gesetzt. Der Block rechnet

```
out = summe_e  down_proj(h_e) * gewicht_e     (+ geteilter Experte)
```

also entfernt `gewicht_e = 0` den Experten vollstaendig aus dem Gemisch. Das ist ein
reiner Vorwaerts-Haken auf `mlp.experts`:

* kein Sicherungsabzug, kein Speicher - bei ganzen Experten waeren es Gigabyte
* wirkt an **jeder** Position, nicht nur dort, wo der Router zufaellig dieselben
  Experten waehlt wie an der Messstelle
* restlos umkehrbar: Haken abnehmen

Dazu das nie angefasste Stueck: der **geteilte Experte** laeuft bei jedem Token mit,
unabhaengig vom Router, und kam in keinem frueheren Kandidatensatz vor. Seine Bauform
liest die Zelle aus, statt sie anzunehmen - fusioniertes `gate_up_proj` oder getrennte
`gate_proj`/`up_proj`; erkennt sie keine von beiden, druckt sie die Formen und
ueberspringt Pruefung II mit Ansage.

## Drei Pruefungen am selben JP-Arm

1. **Routing** - die JP-exklusiven Experten maskieren, gegen gleich viele zufaellige aus
   den *gemeinsamen*. Die Kontrolle hat dieselbe Groesse.
2. **Geteilter Experte** - seine Top-Einheiten nach der JP-NEU-Differenz nullen, gegen
   gleich viele zufaellige aus demselben geteilten Experten.
3. **Leiter** - Anteil aller 256 Experten je Schicht sperren: 1/32, 1/8, 1/2. Nicht
   "K Paare insgesamt": bei 256 Experten je Schicht und 8 gezogenen waere die
   Trefferwahrscheinlichkeit verschwindend, und die Leiter wuerde Dosis vortaeuschen,
   die es nicht gibt.

**Vorregistriert:** 1 und 2 muessen die Japanisch-Rate *senken*, ihre jeweilige
Zufallskontrolle darf es nicht. Die Leiter registriert keine Richtung - sie fragt nur, ab
welcher Dosis sich ueberhaupt etwas bewegt.

| Verdikt | Lesart |
|---|---|
| `ROUTING-TRAEGT` | die Anweisung sitzt in der Router-Auswahl - genau der Menge, die die Eichung ausgeschlossen hatte |
| `GETEILTER-EXPERTE-TRAEGT` | sie sitzt in dem Teil, der bei jedem Token unabhaengig vom Router mitlaeuft |
| `BEIDES-TRAEGT` | beide Wege senken |
| `NUR-STOERUNG` | auch die Kontrolle senkt - Dosis zu hoch |
| `NUR-DOSIS` | nur grossflaechiges Sperren wirkt - die Anweisung ist ueber das Gemisch verteilt |
| `ROBUST` | nichts bewegt sie, auch nicht das Sperren der Haelfte aller Experten |

Nachgeprueft wird bei jedem Eingriff: gesperrte Router-Plaetze gezaehlt, groesster
verbliebener Router-Anteil gemessen, Logit-Wirkung bestaetigt, am Ende die Abweichung zum
Ausgangszustand gegen einen *vor* allen Eingriffen genommenen Bezugspunkt.


In [ ]:
# === PHASE 12 - EXPERTEN STATT EINHEITEN =====================================
# Die Eichung hat die Eichmarke bestaetigt und das Instrument verworfen:
#
#   JP  "each service's Japanese name"   47/48 = 97.9% japanisch
#   NEU "each service's name"             0/48 =  0.0%
#   512 bestgewaehlte Einheiten aus       48/48 = 100.0%   p=1.0000
#   512 zufaellige Einheiten aus          48/48 = 100.0%   p=1.0000
#   Leiter bis 1024 zufaellige            alles still, Antworten fluessig
#
# An der gemessenen Stelle waren 985 der 4096 dort aktiven Einheiten genullt -
# ein Viertel der aktiven Zwischenschicht - und das Verhalten ruehrte sich
# nicht. Die Auswahl war dabei NICHT wertlos: ihre Einheiten bewegten die
# Logits rund dreifach staerker als zufaellige gleicher Zahl (3.66 gegen 1.27).
# Sie hatte Signal, nur keine Verhaltensfolge.
#
# DER DENKFEHLER, DEN DIESER LAUF BEHEBT
#
#   aktive Paare: JP 320 | NEU 320 | in BEIDEN 278
#
# 42 (Schicht,Experte)-Paare feuern im JP-Arm und im NEU-Arm nicht. Der
# Unterschied zwischen "antworte japanisch" und "antworte englisch" liegt zu
# einem guten Teil darin, WELCHE EXPERTEN UEBERHAUPT LAUFEN. Der Eingriff hat
# daran nichts geaendert - im Gegenteil: die Beschraenkung auf gemeinsame
# Paare (der richtige Fix gegen die strukturellen Nullen) hat die 42
# unterscheidenden Experten SYSTEMATISCH AUSGESCHLOSSEN. Gesucht wurde in der
# Schnittmenge, waehrend die Information in der Differenz liegt.
#
# EIN ANDERES INSTRUMENT, NICHT EINE ANDERE AUSWAHL
#
# Statt Gewichtszeilen zu nullen wird der ROUTER-ANTEIL ganzer Experten auf
# null gesetzt. Der Block rechnet
#     out = summe_e  down_proj(h_e) * gewicht_e        (+ geteilter Experte)
# also entfernt gewicht_e = 0 den Experten vollstaendig aus dem Gemisch. Das
# ist ein reiner Vorwaerts-Haken auf 'mlp.experts':
#   - kein Sicherungsabzug, kein Speicher (bei ganzen Experten waeren es GB)
#   - wirkt an JEDER Position, nicht nur dort, wo der Router zufaellig
#     dieselben Experten waehlt wie an der Messstelle
#   - restlos umkehrbar: Haken abnehmen
#
# DAZU DAS NIE ANGEFASSTE STUECK: der GETEILTE EXPERTE laeuft bei jedem Token
# mit, unabhaengig vom Router. Er war in keinem Kandidatensatz der bisherigen
# Laeufe - 512 dichte Einheiten je Schicht, keine strukturellen Nullen.
#
# DREI PRUEFUNGEN AM SELBEN JP-ARM
#  I   die JP-exklusiven Experten maskieren, gegen gleich viele zufaellige
#      aus den GEMEINSAMEN (die Kontrolle muss dieselbe Groesse haben)
#  II  die Top-Einheiten des geteilten Experten nullen, gegen gleich viele
#      zufaellige aus demselben geteilten Experten
#  III Leiter ueber den ANTEIL aller 256 Experten je Schicht: 1/32, 1/8, 1/2 -
#      wie viel vom Gemisch laesst sich loeschen, bevor die Anweisung bricht
#
# VORREGISTRIERT: I und II muessen die Japanisch-Rate SENKEN, ihre jeweilige
# Zufallskontrolle darf es nicht. Die Leiter registriert keine Richtung - sie
# fragt nur, ab welcher Dosis sich ueberhaupt etwas bewegt.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_experten_maske")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
JPW=[(0x3040,0x30FF),(0x3400,0x9FFF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _in(c,bereiche):
    o=ord(c); return any(a<=o<=b for a,b in bereiche)
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250 and _in(c,FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and _in(ch,FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def ist_jp(t,mindest=3):
    """Eigener Zaehler fuer die Positivkontrolle. Im JP-Arm ist japanische
       Schrift das ERWUENSCHTE Verhalten - classify_breit wuerde sie
       'takeover' nennen, was hier irrefuehrend waere. Gezaehlt wird ein Lauf
       von mindestens 3 Kana-/Kanji-Zeichen: einzelne Zeichen kommen auch in
       englischen Antworten als Beispiel vor, ein Lauf nicht."""
    c=0
    for ch in t:
        if _in(ch,JPW):
            c+=1
            if c>=mindest: return True
        elif ch.isalpha(): c=0
    return False
def wiederholt(t,fenster=12,mal=4):
    """Zerfallsmerkmal: dieselbe Zeichenfolge viermal. Bei starker Beschaedigung
       faellt ein Modell in Schleifen, lange bevor es verstummt."""
    if len(t)<fenster*mal: return False
    z=collections.Counter(t[i:i+fenster] for i in range(len(t)-fenster+1))
    return max(z.values())>=mal
def zerfall(texte):
    """Was sagt die Antwortform ueber den Schaden - unabhaengig von der Sprache"""
    if not texte: return dict(leer=0.0,laenge=0.0,schleife=0.0)
    return dict(leer=sum(1 for t in texte if not t.strip())/len(texte),
                laenge=sum(len(t) for t in texte)/len(texte),
                schleife=sum(1 for t in texte if wiederholt(t))/len(texte))
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig: return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def phrase_mit(w):
    return "each service's name" if not w else "each service's %s name"%w
def setze_arm(text,ersatz):
    if PHRASE not in text: return text,False
    return text.replace(PHRASE,ersatz),True
def massstab(*vektoren):
    """EIN globaler, robuster Massstab fuer alle Einheiten. Ersetzt den
       Nenner je Einheit aus v2, der auf 1e-8 fallen konnte und damit
       Trennwerte von 1e7 erzeugt hat. Median statt Mittelwert, weil die
       Zwischenschicht duennbesetzt ist und wenige grosse Werte den Mittelwert
       tragen wuerden. Nullen zaehlen NICHT mit: bei 70% strukturellen Nullen
       waere der Median sonst selbst null."""
    v=np.abs(np.concatenate([np.asarray(x,dtype=np.float64).ravel() for x in vektoren]))
    v=v[v>0]
    if v.size==0: return 1.0
    m=float(np.median(v))
    return m if m>0 else 1.0
def trennung_zwei(xa,xb,s):
    """Differenz zweier Zustaende in Einheiten EINES globalen Massstabs.
       Positiv = im ersten Zustand hoeher. Beschraenkt und vergleichbar."""
    return (np.asarray(xa,dtype=np.float64)-np.asarray(xb,dtype=np.float64))/float(s)
def waehle_einheiten(d,k):
    """die k Einheiten mit der groessten POSITIVEN Trennung - Richtung ist
       vorregistriert: im JP-Zustand hoeher, Ablation muss die JP-Rate senken"""
    return list(np.argsort(-np.asarray(d))[:k])
def zu_einheit(i,paare,breite):
    """flacher Index -> ((Schicht,Experte), Einheit). Die flache Anordnung ist
       genau die aus np.concatenate ueber die Paare in DIESER Reihenfolge."""
    return paare[i//breite],i%breite
def exklusiv(za,zb):
    """Paare, die in za feuern und in zb nicht - genau die Menge, die die
       Eichung ausgeschlossen hatte"""
    return sorted(set(za)-set(zb))
def gemeinsam(za,zb):
    return sorted(set(za)&set(zb))
def nach_schicht(paare):
    """Liste von (Schicht,Experte) -> {Schicht: Menge von Experten}"""
    d=collections.defaultdict(set)
    for l,e in paare: d[int(l)].add(int(e))
    return dict(d)
def anteil_je_schicht(schichten,n_experten,anteil,rnd):
    """Leiterstufe: in JEDER Schicht denselben Anteil aller Experten sperren.
       Nicht 'K Paare insgesamt' - bei 256 Experten je Schicht und 8 gezogenen
       waere die Trefferwahrscheinlichkeit sonst verschwindend, und die Leiter
       wuerde Dosis vortaeuschen, die es nicht gibt."""
    k=int(round(anteil*n_experten))
    return {l:set(rnd.sample(range(n_experten),k)) for l in schichten},k
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    """senkt / hebt / still - zweiseitig, weil die Leiter keine Richtung
       vorregistriert: sie fragt nur, ob sich UEBERHAUPT etwas bewegt"""
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_arm(k_bas,n_bas,k_aus,n_aus,k_zuf,n_zuf,alpha=0.05):
    a=urteil_dosis(k_bas,n_bas,k_aus,n_aus,alpha)=="senkt"
    z=urteil_dosis(k_bas,n_bas,k_zuf,n_zuf,alpha)=="senkt"
    if a and not z: return "TRAEGT"
    if a and z:     return "NUR-STOERUNG"
    if z and not a: return "WIDERSPRUECHLICH"
    return "BLIND"
def urteil_maske(jp_rate,u_routing,u_geteilt,leiter,mindestrate=0.5):
    """Gesamturteil. Der Sanitaetscheck kommt zuerst: ohne gueltige Eichmarke
       gibt es nichts zu senken. u_geteilt darf None sein - der geteilte
       Experte kann fehlen oder anders aufgebaut sein als erwartet."""
    if jp_rate<mindestrate: return "EICHMARKE-FEHLT"
    t1=(u_routing=="TRAEGT"); t2=(u_geteilt=="TRAEGT")
    if t1 and t2: return "BEIDES-TRAEGT"
    if t1: return "ROUTING-TRAEGT"
    if t2: return "GETEILTER-EXPERTE-TRAEGT"
    if "NUR-STOERUNG" in (u_routing,u_geteilt): return "NUR-STOERUNG"
    if any(u!="still" for u in leiter): return "NUR-DOSIS"
    return "ROBUST"
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ARM=int(globals().get("N_ARM",48)); MAX_NEW=int(globals().get("MAX_NEW",64))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260810))
ANTEILE=list(globals().get("ANTEILE",[1/32.,1/8.,1/2.]))
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
ROH_PROMPT=PROMPTS[ZIEL_ID]
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---------------- 0  Architektur --------------------------------------------
print("="*80); print("0  ARCHITEKTUR"); print("="*80)
cfg=model.config
RX=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.experts$")
RXS=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\."
               r"shared_expert$")
EXPM={}; SHM={}
for nm,mod in model.named_modules():
    m=RX.match(nm)
    if m: EXPM[int(m.group(1))]=mod
    m=RXS.match(nm)
    if m: SHM[int(m.group(1))]=mod
ARCH_OK=bool(EXPM)
if ARCH_OK:
    e0=EXPM[min(EXPM)]
    GU=e0.gate_up_proj; DP=e0.down_proj; INTER=int(e0.intermediate_dim)
    NEXP=int(GU.shape[0])
    ARCH_OK=(GU.ndim==3 and DP.ndim==3 and GU.shape[1]==2*INTER
             and GU.shape[2]==cfg.hidden_size and DP.shape[2]==INTER)
    print("  %d Schichten | %d Experten je Schicht | Zwischenbreite %d | top-%d"
          %(len(EXPM),NEXP,INTER,cfg.num_experts_per_tok))
    print("  Formen wie erwartet: %s"%("ja" if ARCH_OK else "NEIN"))
if not ARCH_OK:
    MASKE_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",arch_ok=False)
    wc_save_all(); print(""); print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN"); raise SystemExit(0)
# ---------------- geteilter Experte: entdecken, nicht annehmen ---------------
print(""); print("  GETEILTER EXPERTE - ausgelesen, nicht angenommen")
GT_ART=None; GT_INTER=0
if not SHM:
    print("    kein Modul 'mlp.shared_expert' gefunden - Pruefung II entfaellt")
else:
    s0=SHM[min(SHM)]
    print("    Modul %s"%type(s0).__name__)
    for pn,pv in list(s0.named_parameters())[:6]:
        print("      %-24s %s"%(pn,tuple(pv.shape)))
    if hasattr(s0,"gate_up_proj"):
        W=getattr(s0.gate_up_proj,"weight",s0.gate_up_proj)
        if W.ndim==2 and W.shape[0]%2==0:
            GT_ART="fusioniert"; GT_INTER=int(W.shape[0]//2)
    elif hasattr(s0,"gate_proj") and hasattr(s0,"up_proj"):
        Wg=getattr(s0.gate_proj,"weight",s0.gate_proj)
        Wu=getattr(s0.up_proj,"weight",s0.up_proj)
        if Wg.ndim==2 and Wg.shape==Wu.shape:
            GT_ART="getrennt"; GT_INTER=int(Wg.shape[0])
    print("    Aufbau: %s | Zwischenbreite %d"%(GT_ART or "UNBEKANNT",GT_INTER))
    if GT_ART is None:
        print("    Aufbau nicht erkannt - Pruefung II entfaellt, die Formen stehen oben")
# ---------------- Werkzeuge ---------------------------------------------------
def zieh(text,n,startwert):
    aus=[]
    for b0 in range(0,n,CHUNK):
        b=min(CHUNK,n-b0)
        enc=tokenizer([text]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(startwert+b0)
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            aus.append(tokenizer.decode(g[j,enc["input_ids"].shape[1]:],skip_special_tokens=True))
    return aus
class Maske:
    """Setzt den Router-Anteil ganzer Experten auf null. Der Block rechnet
       out = summe_e down_proj(h_e)*gewicht_e, also entfernt gewicht_e = 0 den
       Experten vollstaendig. Reiner Vorwaerts-Haken: kein Sicherungsabzug,
       kein Speicher, wirkt an JEDER Position, restlos umkehrbar.

       Die Buchfuehrung laeuft nur bei 'pruefen' - waehrend der Erzeugung
       wuerde ein .item() je Schicht und Token die Zelle unnoetig bremsen."""
    def __init__(self,verboten):
        self.verboten={l:set(v) for l,v in verboten.items() if v}
        self.griffe=[]; self.pruefen=False
        self.getroffen=0; self.rest=0.0; self.formen=None
    def _mach(self,bad):
        def h(mod,args):
            idx=args[1]; w=args[2]
            if idx.shape!=w.shape:
                self.formen=(tuple(idx.shape),tuple(w.shape)); return None
            tr=torch.isin(idx,bad)
            neu=w.masked_fill(tr,0.0)
            if self.pruefen:
                self.getroffen+=int(tr.sum().item())
                if bool(tr.any()):
                    self.rest=max(self.rest,float(neu[tr].abs().max().item()))
            return (args[0],idx,neu)+tuple(args[3:])
        return h
    def __enter__(self):
        for l,vs in self.verboten.items():
            W=EXPM[l].gate_up_proj
            bad=torch.tensor(sorted(vs),device=W.device,dtype=torch.long)
            self.griffe.append(EXPM[l].register_forward_pre_hook(self._mach(bad)))
        return self
    def __exit__(self,*a):
        for g in self.griffe: g.remove()
        self.griffe=[]
        return False
def geteilt_zeilen(mod,u):
    """(Gewichtsmatrix, Zeile)-Paare, die Einheit u des geteilten Experten
       abschalten. act_fn(0)*0 = 0 in beiden Aufbauten."""
    if GT_ART=="fusioniert":
        W=getattr(mod.gate_up_proj,"weight",mod.gate_up_proj)
        return [(W,u),(W,GT_INTER+u)]
    Wg=getattr(mod.gate_proj,"weight",mod.gate_proj)
    Wu=getattr(mod.up_proj,"weight",mod.up_proj)
    return [(Wg,u),(Wu,u)]
def geteilt_null(einheiten):
    sicher=[]
    with torch.no_grad():
        for l,u in einheiten:
            for W,r in geteilt_zeilen(SHM[l],u):
                sicher.append((W,r,W[r].clone())); W[r].zero_()
    return sicher
def geteilt_her(sicher):
    with torch.no_grad():
        for W,r,v in sicher: W[r].copy_(v)
def hole_zustand(text,mit_geteilt=True):
    """Routing je Schicht am LETZTEN Token, dazu die Zwischenschicht des
       geteilten Experten. Der letzte Token ist in allen Armen derselbe -
       nur der Zusammenhang davor unterscheidet sich."""
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    fang={}; fang_g={}
    def mach(l):
        def h(mod,args): fang[l]=args[1].detach(); return None
        return h
    def mach_g(l):
        def h(mod,args): fang_g[l]=args[0].detach(); return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    if mit_geteilt and GT_ART:
        hs+=[SHM[l].register_forward_pre_hook(mach_g(l)) for l in SHM]
    try:
        with torch.no_grad():
            o=model(ids[:,:-1],use_cache=True); fang.clear(); fang_g.clear()
            o2=model(ids[:,-1:],past_key_values=o.past_key_values,use_cache=True)
            lg=o2.logits[0,-1].float().cpu().numpy()
    finally:
        for h in hs: h.remove()
    routing=set()
    for l,idx in fang.items():
        for e in idx.reshape(-1,idx.shape[-1])[-1].tolist(): routing.add((l,int(e)))
    geteilt={}
    if mit_geteilt and GT_ART:
        with torch.no_grad():
            for l,x in fang_g.items():
                xv=x.reshape(-1,x.shape[-1])[-1]
                if GT_ART=="fusioniert":
                    W=getattr(SHM[l].gate_up_proj,"weight",SHM[l].gate_up_proj)
                    g,u=torch.nn.functional.linear(xv,W).chunk(2,dim=-1)
                else:
                    Wg=getattr(SHM[l].gate_proj,"weight",SHM[l].gate_proj)
                    Wu=getattr(SHM[l].up_proj,"weight",SHM[l].up_proj)
                    g=torch.nn.functional.linear(xv,Wg); u=torch.nn.functional.linear(xv,Wu)
                geteilt[l]=(SHM[l].act_fn(g)*u).float().cpu().numpy()
    return sorted(routing),geteilt,lg
def nur_logits(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        return model(ids).logits[0,-1].float().cpu().numpy()
def wilson_zeile(nm,k,n,kb,nb,extra=""):
    pp,lo,hi=wilson(k,n)
    pv="-" if kb is None else "%.4f"%fisher2x2(k,n-k,kb,nb-kb)
    print("  %-30s %3d/%-4d %5.1f%% [%4.1f,%4.1f] %9s  %s"
          %(nm,k,n,100*pp,100*lo,100*hi,pv,extra))
def zf(texte):
    z=zerfall(texte)
    return "leer %.0f%% | %3.0f Zeichen | Schleife %.0f%%"%(100*z["leer"],z["laenge"],
                                                            100*z["schleife"])
def mit_maske(verboten,text,n,startwert,pruef_text,pruef_logit):
    """maskieren, nachpruefen, ziehen, Haken IMMER abnehmen"""
    with Maske(verboten) as M:
        M.pruefen=True
        lg=nur_logits(pruef_text)
        M.pruefen=False
        if M.formen is not None:
            raise RuntimeError("Router-Formen passen nicht zueinander: idx %s, "
                               "gewichte %s - die Maske greift nicht"%M.formen)
        wirk=float(np.abs(lg-pruef_logit).max())
        aus=zieh(text,n,startwert)
    return aus,M.getroffen,M.rest,wirk
# ---------------- 1  Eichmarke ------------------------------------------------
print(""); print("="*80); print("1  EICHMARKE"); print("="*80)
JP_ROH,ok_j=setze_arm(ROH_PROMPT,phrase_mit("Japanese"))
NE_ROH,ok_n=setze_arm(ROH_PROMPT,phrase_mit(""))
assert ok_j and ok_n,"Phrase %r nicht im Prompt gefunden"%PHRASE
JP=prompt_text(JP_ROH); NE=prompt_text(NE_ROH)
t0=time.time(); A_JP=zieh(JP,N_ARM,SEED+1); A_NE=zieh(NE,N_ARM,SEED+2)
K_JP=sum(ist_jp(t) for t in A_JP); K_NE=sum(ist_jp(t) for t in A_NE)
print("  (%.0f s)"%(time.time()-t0)); print("")
print("  %-30s %8s %7s %20s %9s"%("Arm","k/n","Rate","95%-Intervall","p vs Basis"))
wilson_zeile("JP  japanische Schrift",K_JP,N_ARM,None,None,zf(A_JP))
wilson_zeile("NEU japanische Schrift",K_NE,N_ARM,None,None,zf(A_NE))
# ---------------- 2  Zustaende ------------------------------------------------
print(""); print("="*80); print("2  ROUTING UND GETEILTER EXPERTE VERGLEICHEN")
R_JP,G_JP,L_JP=hole_zustand(JP); R_NE,G_NE,L_NE=hole_zustand(NE)
EXKL=exklusiv(R_JP,R_NE); GEMEIN=gemeinsam(R_JP,R_NE)
print("  aktive Paare: JP %d | NEU %d | nur JP %d | gemeinsam %d"
      %(len(R_JP),len(R_NE),len(EXKL),len(GEMEIN)))
print("  Schichten der JP-exklusiven: %s"
      %", ".join("L%d:%d"%(l,n) for l,n in sorted(collections.Counter(l for l,_ in EXKL).items())[:16]))
# EIN Bezugspunkt fuer alle Wirkungs- und Wiederherstellungspruefungen. L_JP
# kommt aus dem zweistufigen Lauf mit Cache, nur_logits aus einem einzigen
# Vorwaertspass - die beiden Wege unterscheiden sich numerisch minimal. Wer
# sie mischt, misst diesen Unterschied statt der Wirkung des Eingriffs.
L0=nur_logits(JP)
print("  Bezugslogits (ein Vorwaertspass) | Abstand zum Cache-Lauf %.2e"
      %float(np.abs(L0-L_JP).max()))
assert EXKL,"kein Paar ist JP-exklusiv - dann gibt es keinen Routing-Unterschied zu pruefen"
assert len(GEMEIN)>=len(EXKL),"zu wenige gemeinsame Paare fuer eine gleich grosse Kontrolle"
# ---------------- 3  Pruefung I: Routing --------------------------------------
print(""); print("="*80); print("3  PRUEFUNG I - JP-EXKLUSIVE EXPERTEN MASKIEREN")
print("  vorregistriert: muss SENKEN, die gleich grosse Zufallskontrolle nicht")
rnd=random.Random(SEED)
ZUF_PAARE=rnd.sample(GEMEIN,len(EXKL))
a1,g1,r1,w1=mit_maske(nach_schicht(EXKL),JP,N_ARM,SEED+101,JP,L0)
a2,g2,r2,w2=mit_maske(nach_schicht(ZUF_PAARE),JP,N_ARM,SEED+102,JP,L0)
K_EX=sum(ist_jp(t) for t in a1); K_ZP=sum(ist_jp(t) for t in a2)
print("  NACHPRUEFUNG  gesperrte Router-Plaetze %d / %d | groesster Restanteil "
      "%.2e / %.2e | Logit bewegt %.4f / %.4f"%(g1,g2,r1,r2,w1,w2))
assert max(r1,r2)<1e-6,"maskierter Experte behaelt Router-Gewicht"
assert min(g1,g2)>0,"kein einziger Router-Platz gesperrt - die Maske greift nicht"
assert max(w1,w2)>1e-3,"Maske ohne jede Logit-Wirkung"
print("")
wilson_zeile("ohne Eingriff",K_JP,N_ARM,None,None,zf(A_JP))
wilson_zeile("%d JP-exklusive gesperrt"%len(EXKL),K_EX,N_ARM,K_JP,N_ARM,zf(a1))
wilson_zeile("%d zufaellige gemeinsame"%len(ZUF_PAARE),K_ZP,N_ARM,K_JP,N_ARM,zf(a2))
U_ROUT=urteil_arm(K_JP,N_ARM,K_EX,N_ARM,K_ZP,N_ARM)
print("  -> %s"%U_ROUT)
# ---------------- 4  Pruefung II: geteilter Experte ---------------------------
print(""); print("="*80); print("4  PRUEFUNG II - GETEILTER EXPERTE")
U_GETEILT=None; GT_ZEILE=None
if not GT_ART:
    print("  entfaellt: der geteilte Experte wurde nicht in erwarteter Form gefunden")
elif not G_JP or not G_NE:
    print("  entfaellt: keine Zwischenschicht aufgezeichnet")
else:
    SCHICHTEN=sorted(set(G_JP)&set(G_NE))
    XJ=np.concatenate([G_JP[l] for l in SCHICHTEN]).astype(np.float64)
    XN=np.concatenate([G_NE[l] for l in SCHICHTEN]).astype(np.float64)
    S=massstab(XJ,XN); D=trennung_zwei(XJ,XN,S)
    K_GT=int(globals().get("K_GETEILT",512))
    K_GT=min(K_GT,len(D))
    print("  %d Schichten x %d Einheiten = %d dicht besetzte Einheiten "
          "(keine strukturellen Nullen)"%(len(SCHICHTEN),GT_INTER,len(D)))
    print("  Massstab %.4f | Trennung Median %.3f | Maximum %.3f"
          %(S,float(np.median(D)),float(D.max())))
    idx=waehle_einheiten(D,K_GT)
    AUSW=[zu_einheit(i,SCHICHTEN,GT_INTER) for i in idx]
    aktiv=[i for i in range(len(D)) if max(abs(XJ[i]),abs(XN[i]))>1e-3]
    ZUFG=[zu_einheit(i,SCHICHTEN,GT_INTER) for i in rnd.sample(aktiv,min(K_GT,len(aktiv)))]
    def geteilt_lauf(einheiten,startwert):
        sicher=geteilt_null(einheiten)
        try:
            lg=nur_logits(JP); w=float(np.abs(lg-L0).max())
            _,g2,_=hole_zustand(JP)
            rest=max((float(abs(g2[l][u])) for l,u in einheiten if l in g2),default=0.0)
            aus=zieh(JP,N_ARM,startwert)
        finally:
            geteilt_her(sicher)
        return aus,rest,w
    a3,r3,w3=geteilt_lauf(AUSW,SEED+201)
    a4,r4,w4=geteilt_lauf(ZUFG,SEED+202)
    K_GA=sum(ist_jp(t) for t in a3); K_GZ=sum(ist_jp(t) for t in a4)
    print("  NACHPRUEFUNG  groesster Rest %.2e / %.2e | Logit bewegt %.4f / %.4f"
          %(r3,r4,w3,w4))
    assert max(r3,r4)<1e-6,"Einheit des geteilten Experten nach dem Eingriff nicht null"
    assert max(w3,w4)>1e-3,"Eingriff am geteilten Experten ohne Logit-Wirkung"
    print("")
    wilson_zeile("ohne Eingriff",K_JP,N_ARM,None,None,zf(A_JP))
    wilson_zeile("%d gewaehlte geteilte aus"%K_GT,K_GA,N_ARM,K_JP,N_ARM,zf(a3))
    wilson_zeile("%d zufaellige geteilte aus"%len(ZUFG),K_GZ,N_ARM,K_JP,N_ARM,zf(a4))
    U_GETEILT=urteil_arm(K_JP,N_ARM,K_GA,N_ARM,K_GZ,N_ARM)
    print("  -> %s"%U_GETEILT)
    GT_ZEILE=dict(k_ausw=K_GA,k_zufall=K_GZ,n=N_ARM,k_einheiten=K_GT,urteil=U_GETEILT,
                  massstab=S,trenn_max=float(D.max()),schichten=len(SCHICHTEN),
                  zerfall_ausw=zerfall(a3),zerfall_zufall=zerfall(a4))
# ---------------- 5  Leiter ---------------------------------------------------
print(""); print("="*80); print("5  LEITER - ANTEIL ALLER EXPERTEN JE SCHICHT SPERREN")
print("  nicht 'K Paare insgesamt': bei %d Experten je Schicht und %d gezogenen"
      %(NEXP,cfg.num_experts_per_tok))
print("  waere die Trefferwahrscheinlichkeit verschwindend - die Leiter wuerde")
print("  Dosis vortaeuschen, die es nicht gibt.")
print("")
wilson_zeile("ohne Eingriff",K_JP,N_ARM,None,None,zf(A_JP))
LEITER=[]; LEITER_ZEILEN=[]
rnd2=random.Random(SEED+999)
for f in ANTEILE:
    verb,k=anteil_je_schicht(sorted(EXPM),NEXP,f,rnd2)
    a,g,r,w=mit_maske(verb,JP,N_ARM,SEED+300+k,JP,L0)
    assert r<1e-6,"maskierter Experte behaelt Router-Gewicht"
    k_j=sum(ist_jp(t) for t in a)
    u=urteil_dosis(K_JP,N_ARM,k_j,N_ARM); LEITER.append(u)
    wilson_zeile("%d von %d je Schicht (%.0f%%)"%(k,NEXP,100*f),k_j,N_ARM,K_JP,N_ARM,zf(a))
    print("  %-30s Router-Plaetze gesperrt %d, Logit bewegt %.4f -> %s"%("",g,w,u))
    LEITER_ZEILEN.append(dict(anteil=f,je_schicht=k,k=k_j,n=N_ARM,urteil=u,
                              gesperrt=g,logit=w,zerfall=zerfall(a)))
# ---------------- Wiederherstellung -------------------------------------------
# gegen L0 von VOR allen Eingriffen - ein Vergleich mit einem frisch
# gerechneten Logit waere immer null und wuerde nichts pruefen
ABW=float(np.abs(nur_logits(JP)-L0).max())
print(""); print("  WIEDERHERSTELLUNG: groesste Logit-Abweichung zum Ausgangszustand %.2e"%ABW)
print("  (Haken abgenommen, Gewichtszeilen zurueckgeschrieben)")
assert ABW<1e-2,"Modell nicht sauber wiederhergestellt - alle Zahlen danach waeren verdorben"
# ---------------- Urteil ------------------------------------------------------
CODE=urteil_maske(K_JP/max(N_ARM,1),U_ROUT,U_GETEILT,LEITER)
print(""); print("="*80); print("VERDIKT: %s"%CODE); print("="*80)
if CODE=="EICHMARKE-FEHLT":
    print("  Der JP-Arm folgt der Anweisung ohne Eingriff nicht (%.0f%%). Dann gibt"%(100*K_JP/N_ARM))
    print("  es nichts zu senken und keine Pruefung sagt etwas.")
elif CODE in ("ROUTING-TRAEGT","BEIDES-TRAEGT"):
    print("  Die %d Experten, die NUR im JP-Arm feuern, aus dem Gemisch zu nehmen"%len(EXKL))
    print("  senkt die Japanisch-Rate - gleich viele gemeinsame Experten nicht. Die")
    print("  Anweisung sitzt damit in der ROUTER-AUSWAHL, nicht in der Staerke")
    print("  einzelner Einheiten. Genau diese Menge hatte die Eichung ausgeschlossen.")
elif CODE=="GETEILTER-EXPERTE-TRAEGT":
    print("  Der geteilte Experte traegt sie - der Teil, der bei JEDEM Token")
    print("  unabhaengig vom Router mitlaeuft und in keinem frueheren Kandidatensatz")
    print("  vorkam. Dort liegt, was das Modell fuer alle Token gleich macht.")
elif CODE=="NUR-STOERUNG":
    print("  Auch die Zufallskontrolle senkt. Auf dieser Dosis wirkt Beschaedigung")
    print("  und nicht Auswahl - kleiner ansetzen, bis die Kontrolle still ist.")
elif CODE=="NUR-DOSIS":
    print("  Erst das Sperren eines grossen Anteils aller Experten bewegt etwas,")
    print("  gezieltes Sperren nicht. Die Anweisung ist ueber das Gemisch verteilt")
    print("  und nicht an einer benennbaren Teilmenge festgemacht.")
else:
    print("  Weder die JP-exklusiven Experten noch der geteilte Experte noch das")
    print("  Sperren von bis zu %d%% aller Experten je Schicht bewegen die Rate."%(100*max(ANTEILE)))
    print("  Dann ist das Verhalten gegen Eingriffe dieser Art unempfindlich - und")
    print("  die Frage braucht ein Instrument, das nicht durch Wegnehmen arbeitet.")
MASKE_RESULTS=dict(verdict=CODE,arch_ok=True,prompt_id=ZIEL_ID,inter=INTER,n_experten=NEXP,
    n_arm=N_ARM,k_jp=K_JP,k_neu=K_NE,zerfall_jp=zerfall(A_JP),zerfall_neu=zerfall(A_NE),
    paare_jp=len(R_JP),paare_neu=len(R_NE),exklusiv=[list(q) for q in EXKL],
    gemeinsam=len(GEMEIN),k_exklusiv=K_EX,k_zufallspaare=K_ZP,urteil_routing=U_ROUT,
    geteilt_art=GT_ART,geteilt_inter=GT_INTER,geteilt=GT_ZEILE,urteil_geteilt=U_GETEILT,
    anteile=ANTEILE,leiter=LEITER_ZEILEN,abweichung_ende=ABW)
wc_save("antworten_maske",dict(prompt_id=ZIEL_ID,jp=A_JP,neu=A_NE,exklusiv=a1,zufall=a2))
wc_save_all()
print("")
print("(Der Eingriff ist reversibel und wurde nachgeprueft: gesperrte Router-Plaetze")
print(" gezaehlt, Restanteil gemessen, Logit-Wirkung bestaetigt, Haken abgenommen.)")
